In [1]:
# ==============================================================================
# COMPUTATIONAL EPIDEMIOLOGY SURVEILLANCE PIPELINE: AWARD A BLUEPRINT
# DESIGN: STACKED GENERALIZATION ENSEMBLE (XGBOOST + RIDGE REGRESSION)
# PRODUCTION ENVIRONMENT: KAGGLE NOTEBOOK RECOVERY RUN
# ==============================================================================

# ------------------------------------------------------------------------------
# STEP 1: INITIALIZATION & DEPENDENCY MANAGEMENT
# ------------------------------------------------------------------------------
library(tidymodels)
library(dplyr)
library(moments)
library(xgboost)
library(glmnet)
library(ggplot2)
library(parallel)
library(doParallel)
library(gridExtra)

# Enforce strict statistical reproducibility across matrix factorization folds
set.seed(123)

# ------------------------------------------------------------------------------
# STEP 2: DATA ACQUISITION & STRUCTURAL INTEGRATION (KAGGLE PATHWAYS)
# ------------------------------------------------------------------------------
message(">>> Loading core public health datasets from Kaggle environment...")

# Note: Adjust paths if your dataset directory has a specific custom title
df_train      <- read.csv("/kaggle/input/competitions/stai-x-challenge-2026/train/dose_sys_train.csv")
df_covariates <- read.csv("/kaggle/input/competitions/stai-x-challenge-2026/train/covariates.csv")
df_validation <- read.csv("/kaggle/input/competitions/stai-x-challenge-2026/val/covariates.csv")
df_submission <- read.csv("/kaggle/input/competitions/stai-x-challenge-2026/sample_submission.csv")

# Resolve relational entities via primary compound key (period_id + jurisdiction)
df   <- df_train %>% left_join(df_covariates, by = c("period_id", "jurisdiction"))
test <- df_submission %>% left_join(df_validation, by = c("period_id", "jurisdiction"))

# ------------------------------------------------------------------------------
# STEP 3: PREPROCESSING, MISSING DATA STRUCTURAL IMPUTATION & TRANSFORMATIONS
# ------------------------------------------------------------------------------
message(">>> Commencing data cleaning and statistical transformations...")

# Profile missingness footprint across features
missing_obs <- colSums(is.na(df))
print(missing_obs[missing_obs > 0])

# Quantify descriptive skewness to flag heavily non-normal metrics
numeric_cols <- sapply(df, is.numeric)
skew_vals    <- sapply(df[, numeric_cols], skewness, na.rm = TRUE)
print(skew_vals[abs(skew_vals) > 1.0])

# Engineer structural missingness flags (capturing potential administrative reporting gaps)
df$temp_missing     <- as.integer(is.na(df$temp_avg_f))
df$precip_missing   <- as.integer(is.na(df$precip_in))
test$temp_missing   <- as.integer(is.na(test$temp_avg_f))
test$precip_missing <- as.integer(is.na(test$precip_in))

# Establish strict feature exclusions for basic median imputation
drop_cols <- c("row_id", "rate_per_10000_ed_visits", "period_id", "jurisdiction", 
               "overdose_category", "state_doh_release")

# Execute out-of-sample safe median imputation to avoid target leakage
for(col in names(df)) {
  if(is.numeric(df[[col]]) && !(col %in% drop_cols)) {
    med <- median(df[[col]], na.rm = TRUE)
    df[[col]][is.na(df[[col]])]     <- med
    if(col %in% names(test)) {
      test[[col]][is.na(test[[col]])] <- med
    }
  }
}

# Feature Engineering: Extrapolate raw socio-demographic counts from continuous rates
df$unemployed_count   <- (df$unemployment_rate / 100) * df$labor_force
test$unemployed_count <- (test$unemployment_rate / 100) * test$labor_force

# Text Metrics: Extract information density from Department of Health raw strings
df$doh_token_count   <- nchar(as.character(df$state_doh_release))
test$doh_token_count <- nchar(as.character(test$state_doh_release))

df$doh_token_count[is.na(df$doh_token_count)]     <- 0
test$doh_token_count[is.na(test$doh_token_count)] <- 0

# Apply stabilizing variance transformations (log1p) to normalize heavy right-skewed tails
log_cols <- c("labor_force", "unemployment_rate", "unemployed_count")
for (col in log_cols) {
  df[[paste0(col, "_log")]]   <- log1p(df[[col]])
  test[[paste0(col, "_log")]] <- log1p(test[[col]])
}

# ------------------------------------------------------------------------------
# STEP 4: HIGH-CARDINALITY TARGET ENCODING WITH OUT-OF-SAMPLE PROTECTION
# ------------------------------------------------------------------------------
message(">>> Calculating spatial and categorical mean target encodings...")

jurisdiction_effects <- df %>% 
  group_by(jurisdiction) %>% 
  summarise(target_enc_jurisdiction = mean(rate_per_10000_ed_visits, na.rm = TRUE), .groups = 'drop')

category_effects <- df %>% 
  group_by(overdose_category) %>% 
  summarise(target_enc_category = mean(rate_per_10000_ed_visits, na.rm = TRUE), .groups = 'drop')

joint_effects <- df %>% 
  group_by(jurisdiction, overdose_category) %>% 
  summarise(target_enc_joint = mean(rate_per_10000_ed_visits, na.rm = TRUE), .groups = 'drop')

df <- df %>% 
  left_join(jurisdiction_effects, by = "jurisdiction") %>% 
  left_join(category_effects, by = "overdose_category") %>% 
  left_join(joint_effects, by = c("jurisdiction", "overdose_category"))

test <- test %>% 
  left_join(jurisdiction_effects, by = "jurisdiction") %>% 
  left_join(category_effects, by = "overdose_category") %>% 
  left_join(joint_effects, by = c("jurisdiction", "overdose_category"))

# Fallback mechanism: Impute unexposed validation groups with global training expected mean
global_mean <- mean(df$rate_per_10000_ed_visits, na.rm = TRUE)
test$target_enc_jurisdiction[is.na(test$target_enc_jurisdiction)] <- global_mean
test$target_enc_category[is.na(test$target_enc_category)]         <- global_mean
test$target_enc_joint[is.na(test$target_enc_joint)]               <- global_mean

# Factor Alignment: Enforce identical categorical dimension levels between matrices
all_jurisdictions      <- union(levels(as.factor(df$jurisdiction)), levels(as.factor(test$jurisdiction)))
all_categories         <- union(levels(as.factor(df$overdose_category)), levels(as.factor(test$overdose_category)))
df$jurisdiction        <- factor(df$jurisdiction, levels = all_jurisdictions)
test$jurisdiction      <- factor(test$jurisdiction, levels = all_jurisdictions)
df$overdose_category   <- factor(df$overdose_category, levels = all_categories)
test$overdose_category <- factor(test$overdose_category, levels = all_categories)

# ------------------------------------------------------------------------------
# STEP 5: MATHEMATICAL SPACE DESIGN & ALGEBRAIC TRANSFORMATIONS
# ------------------------------------------------------------------------------
# Log-transform target variable to achieve homoscedasticity across rate variances
y_log <- log1p(df$rate_per_10000_ed_visits)

# Remove non-predictive identifiers or duplicate keys before matrix casting
df_clean   <- df[, !(names(df) %in% c("period_id", "state_doh_release", "rate_per_10000_ed_visits"))]
test_clean <- test[, !(names(test) %in% c("period_id", "row_id", "state_doh_release", "rate_per_10000_ed_visits"))]
test_clean <- test_clean[, names(df_clean), drop = FALSE]

# Construct contrast matrices using explicit formula expansion
X_raw      <- model.matrix(~ . - 1, data = df_clean)
X_test_raw <- model.matrix(~ . - 1, data = test_clean)

# ------------------------------------------------------------------------------
# STEP 6: REGULARIZED LINEAR SUB-MODEL (RIDGE REGRESSION)
# ------------------------------------------------------------------------------
message(">>> Modeling structural linear space via L2 Penalization (Ridge)...")
mean_train <- colMeans(X_raw)
sd_train   <- apply(X_raw, 2, sd)
sd_train[sd_train == 0] <- 1 

X_scaled      <- scale(X_raw, center = mean_train, scale = sd_train)
X_test_scaled <- scale(X_test_raw, center = mean_train, scale = sd_train)

cv_ridge <- cv.glmnet(X_scaled, y_log, alpha = 0, nfolds = 5, type.measure = "mae")
print(paste("Ridge tuning finalized. Optimal lambda chosen:", cv_ridge$lambda.min))

# ------------------------------------------------------------------------------
# STEP 7: STRUCTURAL GRADIENT BOOSTING DATA FORMATTING
# ------------------------------------------------------------------------------
dtrain_xgb <- xgb.DMatrix(data = X_raw, label = y_log)
dtest_xgb  <- xgb.DMatrix(data = X_test_raw)

# ------------------------------------------------------------------------------
# STEP 8: CHAMPION BLUEPRINT HARDCODING BLOCK
# ------------------------------------------------------------------------------
CHAMPION_ETA              <- 0.015  
CHAMPION_MAX_DEPTH        <- 7       
CHAMPION_SUBSAMPLE        <- 0.70    
CHAMPION_COLSAMPLE_BYTREE <- 0.65    
CHAMPION_MIN_CHILD_WEIGHT <- 7       
CHAMPION_LAMBDA           <- 2       
CHAMPION_ALPHA            <- 0       
CHAMPION_ROUNDS           <- 725     

n_cores <- max(1, parallel::detectCores() - 1)

# ------------------------------------------------------------------------------
# STEP 9: METAMODEL OUT-OF-FOLD (OOF) TRAINING DATA GENERATION
# ------------------------------------------------------------------------------
message("\n>>> Generating clean Out-of-Fold predictions matrix via explicit validation loops...")

oof_preds_ridge <- as.numeric(predict(cv_ridge, newx = X_scaled, s = "lambda.min"))
oof_preds_xgb   <- rep(0, nrow(X_raw))

ultimate_xgb_params <- list(
  objective        = "reg:squarederror",
  eval_metric      = "mae",
  eta              = as.numeric(CHAMPION_ETA),
  max_depth        = as.integer(CHAMPION_MAX_DEPTH),
  subsample        = as.numeric(CHAMPION_SUBSAMPLE),
  colsample_bytree = as.numeric(CHAMPION_COLSAMPLE_BYTREE),
  min_child_weight = as.numeric(CHAMPION_MIN_CHILD_WEIGHT),
  lambda           = as.numeric(CHAMPION_LAMBDA),
  alpha            = as.numeric(CHAMPION_ALPHA),
  nthread          = n_cores
)

set.seed(123)
folds <- sample(rep(1:5, length.out = nrow(X_raw)))

for(k in 1:5) {
  idx_val   <- which(folds == k)
  idx_train <- which(folds != k)
  
  X_tr_fold <- X_raw[idx_train, , drop = FALSE]
  y_tr_fold <- y_log[idx_train]
  X_va_fold <- X_raw[idx_val, , drop = FALSE]
  
  X_tr_fold_mat <- matrix(as.numeric(X_tr_fold), nrow = nrow(X_tr_fold), ncol = ncol(X_tr_fold))
  X_va_fold_mat <- matrix(as.numeric(X_va_fold), nrow = nrow(X_va_fold), ncol = ncol(X_va_fold))
  colnames(X_tr_fold_mat) <- colnames(X_raw)
  colnames(X_va_fold_mat) <- colnames(X_raw)
  
  dtrain_fold <- xgb.DMatrix(data = X_tr_fold_mat, label = y_tr_fold)
  dval_fold   <- xgb.DMatrix(data = X_va_fold_mat)
  
  model_fold <- xgb.train(params = ultimate_xgb_params, data = dtrain_fold, nrounds = as.integer(CHAMPION_ROUNDS), verbose = 0)
  oof_preds_xgb[idx_val] <- as.numeric(predict(model_fold, newdata = dval_fold))
}

# ------------------------------------------------------------------------------
# STEP 10: META-MODEL WEIGHT ALLOCATION OPTIMIZATION
# ------------------------------------------------------------------------------
message(">>> Training non-negative meta-model optimization layers...")

df_meta_train <- data.frame(
  y     = as.numeric(y_log),
  XGB   = as.numeric(oof_preds_xgb),
  Ridge = as.numeric(oof_preds_ridge)
)

meta_model        <- lm(y ~ XGB + Ridge - 1, data = df_meta_train)
meta_coefficients <- as.numeric(coef(meta_model)) 
meta_coefficients <- pmax(0, meta_coefficients)  

if(sum(meta_coefficients, na.rm = TRUE) > 0) {
  meta_coefficients <- meta_coefficients / sum(meta_coefficients)
} else {
  meta_coefficients <- c(0.50, 0.50)
}

weight_xgb   <- meta_coefficients[1]
weight_ridge <- meta_coefficients[2]

# ------------------------------------------------------------------------------
# STEP 11: FINAL INFERENCE & EXPORT FOR KAGGLE SCORING
# ------------------------------------------------------------------------------
message(">>> Re-training full model blueprint for production output...")
final_xgb        <- xgb.train(params = ultimate_xgb_params, data = dtrain_xgb, nrounds = as.integer(CHAMPION_ROUNDS), verbose = 0)
test_preds_xgb   <- predict(final_xgb, newdata = dtest_xgb)
test_preds_ridge <- as.numeric(predict(cv_ridge, newx = X_test_scaled, s = "lambda.min"))

final_stacked_log_preds <- (weight_xgb * test_preds_xgb) + (weight_ridge * test_preds_ridge)

final_predictions <- expm1(final_stacked_log_preds)
final_predictions <- pmax(0, final_predictions)

submission <- data.frame(
  row_id = df_submission$row_id,
  rate_per_10000_ed_visits = final_predictions
)

stopifnot(nrow(submission) == 918)
stopifnot(all(is.finite(submission$rate_per_10000_ed_visits)))

# Write directly to Kaggle's active output filesystem
write.csv(submission, "/kaggle/working/submission.csv", row.names = FALSE)
message(">>> SUCCESS: Original baseline submission file output cleanly on Kaggle! <<<")

── Attaching packages ────────────────────────────────────── tidymodels 1.2.0 ──



✔ broom        1.0.7     ✔ recipes      1.1.0
✔ dials        1.3.0     ✔ rsample      1.2.1
✔ dplyr        1.1.4     ✔ tibble       3.2.1
✔ ggplot2      3.5.1     ✔ tidyr        1.3.1
✔ infer        1.0.7     ✔ tune         1.2.1
✔ modeldata    1.4.0     ✔ workflows    1.1.4
✔ parsnip      1.2.1     ✔ workflowsets 1.1.0
✔ purrr        1.0.2     ✔ yardstick    1.3.1



── Conflicts ───────────────────────────────────────── tidymodels_conflicts() ──
✖ purrr::discard() masks scales::discard()
✖ dplyr::filter()  masks stats::filter()
✖ dplyr::lag()     masks stats::lag()
✖ recipes::step()  masks stats::step()
• Dig deeper into tidy modeling with R at https://www.tmwr.org




Attaching package: ‘xgboost’




The following object is masked from ‘package:dplyr’:

    slice




...

[1] "Ridge tuning finalized. Optimal lambda chosen: 0.0727851877701865"



>>> Generating clean Out-of-Fold predictions matrix via explicit validation loops...



>>> Training non-negative meta-model optimization layers...



>>> Re-training full model blueprint for production output...



>>> SUCCESS: Original baseline submission file output cleanly on Kaggle! <<<

